# 1. A model on its own

**LangGraph tutorial, lesson 1 of 5**

Before building an agent, it is worth seeing the ceiling you are about to break
through. A language model is a text-in, text-out function. It has:

- **no clock** — it cannot tell you today's date
- **no database** — it knows nothing about your company
- **no calculator** — it predicts digits rather than computing them

This notebook demonstrates each gap. Everything that follows exists to close them.

## Setup

`make_llm()` lives in `common.py` and wraps `ChatDatabricks`, which authenticates
automatically inside a Databricks workspace — no API keys or tokens needed.

> If the import fails, run `%pip install langgraph databricks-langchain` in a cell
> above, then restart Python.

In [ ]:
from common import make_llm, text_of

llm = make_llm()
print(f"Using: {type(llm).__name__}")

## What the model CAN do unaided

General knowledge it absorbed during training. No outside help required.

In [ ]:
reply = llm.invoke("In one sentence, what is a directed acyclic graph?")
print(text_of(reply))

## Gap 1: it has no clock

Ask for today's date and there is no mechanism for the model to consult. A
well-behaved model says so; a poorly-behaved one invents a plausible date.

**Neither is useful.** We need to hand it a real clock.

In [ ]:
reply = llm.invoke("What is today's date? Answer with just the date.")
print(text_of(reply))

## Gap 2: arithmetic is expensive, not free

This model is actually *good* at arithmetic — but look at **how** it gets there.

Modern reasoning models are given a private "thinking" budget: tokens spent
working before answering. Watch the token count on a large multiplication.

In [ ]:
QUESTION = "Compute 48712031 * 92842377. Reply with ONLY the number."
TRUTH = 48712031 * 92842377

reply = make_llm(max_tokens=3000).invoke(QUESTION)
usage = reply.usage_metadata

print(f"Model answer : {text_of(reply).strip()}")
print(f"True answer  : {TRUTH}")
print(f"Output tokens: {usage['output_tokens']}")

It got it right — by doing long multiplication in longhand, spending well over a
thousand tokens to do it.

Compare that to the same result in Python:

In [ ]:
print(48712031 * 92842377)   # exact, instant, free

Two different problems, one solution:

| Gap | Why reasoning cannot fix it |
|---|---|
| The date, your database | The information simply is not there |
| Arithmetic | It can, but slowly, expensively, and it is the wrong tool |

Both are solved by letting the model **call real code**. That mechanism is *tool
calling* — lesson 2.

---
**Next:** `02_tool_calling.ipynb`